# BanglaBERT + FGM Training - Binary Datasets (Single Run, Loop Version)

This notebook trains and evaluates **BanglaBERT + FGM adversarial fine-tuning** on all three binary datasets in a single looped pipeline.

What it does:
- trains **BanglaBERT + FGM** on `ben_sarc_binary`, `banglasarc_binary`, and `banglasarc3_binary`
- evaluates on validation and test splits
- saves predictions, probabilities, classification reports, confusion matrices, and run metadata
- creates clean outputs for later comparison, calibration, cross-dataset, and statistics notebooks

Expected input:
- split CSV files from `01_data/interim/splits`

Main saved outputs:
- `04_outputs/tables/06_fgm_banglabert_binary_summary_all_datasets.csv`
- `04_outputs/tables/06_fgm_banglabert_binary_confusion_matrices_all_datasets.json`
- `04_outputs/reports/06_fgm_banglabert_binary_classification_reports_all_datasets.csv`
- `04_outputs/run_logs/06_fgm_banglabert_run_metadata.csv`
- per-dataset prediction files under `04_outputs/predictions/`


In [1]:
from pathlib import Path
import json
import math
import random
import shutil
import time
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import Dataset
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    classification_report,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

print("Torch version:", torch.__version__)
try:
    import transformers
    print("Transformers version:", transformers.__version__)
except Exception as e:
    print("Could not read transformers version:", e)

print("CUDA available:", torch.cuda.is_available())
print("MPS available:", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())

from tqdm.auto import tqdm


/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.11.0
Transformers version: 5.3.0
CUDA available: False
MPS available: True


In [2]:

# =========================
# CONFIG
# =========================

SEED = 42
MODEL_NAME = "csebuetnlp/banglabert"

MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

EPSILON = 0.5
FGM_EMBEDDING_NAME = "word_embeddings"

USE_NORMALIZATION = False
SAVE_VALIDATION_PREDICTIONS = True
SAVE_TEST_PREDICTIONS = True

DATASETS = [
    "ben_sarc_binary",
    "banglasarc_binary",
    "banglasarc3_binary",
]

print("Configured datasets:", DATASETS)
print("FGM epsilon:", EPSILON)


Configured datasets: ['ben_sarc_binary', 'banglasarc_binary', 'banglasarc3_binary']
FGM epsilon: 0.5


In [3]:

# =========================
# REPRODUCIBILITY
# =========================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Seed fixed to:", SEED)


Seed fixed to: 42


In [4]:

# =========================
# PATHS
# =========================

ROOT = Path("..")
SPLITS = ROOT / "01_data" / "interim" / "splits"
TABLES = ROOT / "04_outputs" / "tables"
REPORTS = ROOT / "04_outputs" / "reports"
PREDICTIONS = ROOT / "04_outputs" / "predictions"
RUN_LOGS = ROOT / "04_outputs" / "run_logs"
CHECKPOINTS = ROOT / "03_models" / "checkpoints"

for path in [TABLES, REPORTS, PREDICTIONS, RUN_LOGS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

for dataset_name in DATASETS:
    for split_name in ["train", "val", "test"]:
        split_path = SPLITS / f"{dataset_name}_{split_name}.csv"
        if not split_path.exists():
            raise FileNotFoundError(f"Missing split file: {split_path}")

print("All required split files found.")


All required split files found.


In [5]:

# =========================
# OPTIONAL NORMALIZATION
# =========================

def normalize_text(text):
    # Keep minimal and safe unless you intentionally replace this with the full BanglaBERT normalization pipeline later.
    text = "" if pd.isna(text) else str(text)
    text = " ".join(text.split())
    return text


In [6]:

# =========================
# TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded:", MODEL_NAME)


Tokenizer loaded: csebuetnlp/banglabert


In [7]:

# =========================
# HELPERS
# =========================

def softmax_np(x):
    x = np.asarray(x)
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / exp_x.sum(axis=1, keepdims=True)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_binary": p_bin,
        "recall_binary": r_bin,
        "f1_binary": f1_bin,
        "macro_f1": f1_macro,
    }

def build_eval_table(df_raw, pred_output, dataset_name, split_name):
    logits = pred_output.predictions
    labels = pred_output.label_ids
    probs = softmax_np(logits)
    preds = np.argmax(logits, axis=-1)
    conf = probs.max(axis=1)

    out_df = df_raw.reset_index(drop=True).copy()
    out_df["gold_label"] = labels
    out_df["pred_label"] = preds
    out_df["prob_label_0"] = probs[:, 0]
    out_df["prob_label_1"] = probs[:, 1]
    out_df["confidence"] = conf
    out_df["correct"] = (out_df["gold_label"] == out_df["pred_label"]).astype(int)
    out_df["dataset"] = dataset_name
    out_df["split"] = split_name
    out_df["model"] = "banglabert_fgm"
    out_df["model_name"] = MODEL_NAME
    out_df["epsilon"] = EPSILON
    out_df["seed"] = SEED
    out_df["max_length"] = MAX_LENGTH
    out_df["use_normalization"] = USE_NORMALIZATION
    return out_df

def metrics_from_predictions(dataset_name, split_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    return {
        "model": "banglabert_fgm",
        "model_name": MODEL_NAME,
        "dataset": dataset_name,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "max_length": MAX_LENGTH,
        "seed": SEED,
        "epsilon": EPSILON,
        "fgm_embedding_name": FGM_EMBEDDING_NAME,
        "use_normalization": USE_NORMALIZATION,
        "split": split_name,
        "accuracy": acc,
        "precision_binary": p_bin,
        "recall_binary": r_bin,
        "f1_binary": f1_bin,
        "macro_f1": f1_macro,
        "n_examples": len(y_true),
    }

def label_stats(df, dataset_name, split_name):
    vc = df["label_binary"].value_counts().to_dict()
    n = len(df)
    return {
        "dataset": dataset_name,
        "split": split_name,
        "n_rows": n,
        "label_0_count": vc.get(0, 0),
        "label_0_pct": round(100 * vc.get(0, 0) / n, 2),
        "label_1_count": vc.get(1, 0),
        "label_1_pct": round(100 * vc.get(1, 0) / n, 2),
    }


def detect_text_column(df):
    candidates = ["text", "sentence", "content", "comment", "utterance"]
    for c in candidates:
        if c in df.columns:
            return c
    object_cols = [c for c in df.columns if df[c].dtype == "object"]
    if object_cols:
        return object_cols[0]
    raise KeyError(f"No text-like column found. Available columns: {list(df.columns)}")

def detect_label_column(df):
    candidates = ["label", "labels", "label_binary", "binary_label", "target", "y"]
    for c in candidates:
        if c in df.columns:
            return c
    int_like = []
    for c in df.columns:
        s = df[c]
        try:
            vals = set(pd.Series(s).dropna().unique().tolist())
            if vals and vals.issubset({0,1}):
                int_like.append(c)
        except Exception:
            pass
    if int_like:
        return int_like[0]
    raise KeyError(f"No binary label column found. Available columns: {list(df.columns)}")

def prepare_binary_df(df):
    text_col = detect_text_column(df)
    label_col = detect_label_column(df)
    out = df.copy()
    out["text"] = out[text_col].astype(str)
    out["label"] = out[label_col].astype(int)
    return out, text_col, label_col

def label_stats(df, dataset_name, split_name):
    _, text_col, label_col = prepare_binary_df(df)
    vc = df[label_col].value_counts().to_dict()
    n = len(df)
    return {
        "dataset": dataset_name,
        "split": split_name,
        "text_column": text_col,
        "label_column": label_col,
        "n_rows": n,
        "label_0_count": vc.get(0, 0),
        "label_0_pct": round(100 * vc.get(0, 0) / n, 2) if n else 0.0,
        "label_1_count": vc.get(1, 0),
        "label_1_pct": round(100 * vc.get(1, 0) / n, 2) if n else 0.0,
    }


In [8]:

# =========================
# FGM + TRAINING HELPERS
# =========================

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

class FGM:
    def __init__(self, model, epsilon=0.5, emb_name="word_embeddings"):
        self.model = model
        self.epsilon = epsilon
        self.emb_name = emb_name
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None and self.emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

def make_dataloader(hf_dataset, shuffle=False):
    columns = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in hf_dataset.column_names:
        columns.insert(2, "token_type_ids")
    ds = hf_dataset.with_format("torch", columns=columns)

    use_pin = (str(DEVICE) == "cuda")
    workers = 2 if str(DEVICE) != "cpu" else 0

    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=use_pin,
    )

def batch_to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

@torch.no_grad()
def predict_model(model, dataloader):
    model.eval()
    all_logits = []
    all_labels = []

    for batch in dataloader:
        batch = batch_to_device(batch)
        labels = batch["labels"]
        outputs = model(**batch)
        logits = outputs.logits

        all_logits.append(logits.detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

    logits = np.concatenate(all_logits, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    metrics = {
        "accuracy": acc,
        "precision_binary": p_bin,
        "recall_binary": r_bin,
        "f1_binary": f1_bin,
        "macro_f1": f1_macro,
    }

    return logits, labels, metrics

def train_one_dataset_with_fgm(model, train_loader, val_loader, epochs=3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, len(train_loader) * epochs)
    warmup_steps = int(WARMUP_RATIO * total_steps)

    def lr_lambda(current_step):
        if warmup_steps > 0 and current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        return max(
            0.0,
            float(total_steps - current_step) / float(max(1, total_steps - warmup_steps))
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    fgm = FGM(model, epsilon=EPSILON, emb_name=FGM_EMBEDDING_NAME)

    history = []
    best_state = None
    best_epoch = None
    best_val_macro_f1 = -1.0
    global_step = 0

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False):
            batch = batch_to_device(batch)

            optimizer.zero_grad()

            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            fgm.attack()
            outputs_adv = model(**batch)
            loss_adv = outputs_adv.loss
            loss_adv.backward()
            fgm.restore()

            optimizer.step()
            scheduler.step()

            epoch_loss += float(loss.detach().cpu().item())
            global_step += 1

        avg_train_loss = epoch_loss / max(1, len(train_loader))
        _, _, val_metrics = predict_model(model, val_loader)

        history.append({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_f1_binary": val_metrics["f1_binary"],
        })

        print(
            f"Epoch {epoch}/{epochs} | "
            f"train_loss={avg_train_loss:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
            f"val_f1_binary={val_metrics['f1_binary']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_epoch, best_val_macro_f1, pd.DataFrame(history)


Using device: mps


In [9]:

# =========================
# NOTE
# =========================
# This notebook uses a plain PyTorch training loop with FGM instead of Hugging Face Trainer.
# That avoids local-environment incompatibilities with different transformers versions.


In [10]:

# =========================
# RUN ALL DATASETS
# =========================

all_summary_rows = []
all_confusions = {}
all_report_rows = []
all_label_stats = []
all_run_logs = []

for dataset_name in DATASETS:
    print("\n" + "=" * 100)
    print(f"RUNNING FGM DATASET: {dataset_name}")
    print("=" * 100)

    train_path = SPLITS / f"{dataset_name}_train.csv"
    val_path = SPLITS / f"{dataset_name}_val.csv"
    test_path = SPLITS / f"{dataset_name}_test.csv"

    train_df_raw = pd.read_csv(train_path)
    val_df_raw = pd.read_csv(val_path)
    test_df_raw = pd.read_csv(test_path)

    print(f"Loaded shapes -> train: {train_df_raw.shape} | val: {val_df_raw.shape} | test: {test_df_raw.shape}")

    all_label_stats.append(label_stats(train_df_raw, dataset_name, "train"))
    all_label_stats.append(label_stats(val_df_raw, dataset_name, "val"))
    all_label_stats.append(label_stats(test_df_raw, dataset_name, "test"))

    work_train, train_text_col, train_label_col = prepare_binary_df(train_df_raw)
    work_val, val_text_col, val_label_col = prepare_binary_df(val_df_raw)
    work_test, test_text_col, test_label_col = prepare_binary_df(test_df_raw)

    print(f"Detected columns -> train: text='{train_text_col}', label='{train_label_col}' | "
          f"val: text='{val_text_col}', label='{val_label_col}' | "
          f"test: text='{test_text_col}', label='{test_label_col}'")

    if USE_NORMALIZATION:
        work_train["text"] = work_train["text"].map(normalize_text)
        work_val["text"] = work_val["text"].map(normalize_text)
        work_test["text"] = work_test["text"].map(normalize_text)

    train_ds = Dataset.from_pandas(work_train[["text", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
    val_ds = Dataset.from_pandas(work_val[["text", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
    test_ds = Dataset.from_pandas(work_test[["text", "label"]].rename(columns={"label": "labels"}), preserve_index=False)

    train_tok = train_ds.map(tokenize_batch, batched=True)
    val_tok = val_ds.map(tokenize_batch, batched=True)
    test_tok = test_ds.map(tokenize_batch, batched=True)

    train_loader = make_dataloader(train_tok, shuffle=True)
    val_loader = make_dataloader(val_tok, shuffle=False)
    test_loader = make_dataloader(test_tok, shuffle=False)

    checkpoint_dir = CHECKPOINTS / f"06_fgm_{dataset_name}"
    if checkpoint_dir.exists():
        shutil.rmtree(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    ).to(DEVICE)

    print("Starting FGM training...")
    train_start = time.time()
    model, best_epoch, best_val_macro_f1, history_df = train_one_dataset_with_fgm(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
    )
    train_runtime_sec = time.time() - train_start

    history_path = RUN_LOGS / f"06_fgm_{dataset_name}_epoch_history.csv"
    history_df.to_csv(history_path, index=False)

    # Save checkpoint in a portable way
    model.save_pretrained(checkpoint_dir)
    tokenizer.save_pretrained(checkpoint_dir)

    val_logits, val_labels, val_metrics = predict_model(model, val_loader)
    test_logits, test_labels, test_metrics = predict_model(model, test_loader)

    val_pred_output = type("PredOutput", (), {"predictions": val_logits, "label_ids": val_labels})
    test_pred_output = type("PredOutput", (), {"predictions": test_logits, "label_ids": test_labels})

    val_eval_df = build_eval_table(work_val, val_pred_output, dataset_name, "val")
    test_eval_df = build_eval_table(work_test, test_pred_output, dataset_name, "test")

    if SAVE_VALIDATION_PREDICTIONS:
        val_pred_path = PREDICTIONS / f"06_fgm_banglabert_{dataset_name}_validation_predictions.csv"
        val_eval_df.to_csv(val_pred_path, index=False)

    if SAVE_TEST_PREDICTIONS:
        test_pred_path = PREDICTIONS / f"06_fgm_banglabert_{dataset_name}_test_predictions.csv"
        test_eval_df.to_csv(test_pred_path, index=False)

    all_summary_rows.append({
        "dataset": dataset_name,
        "split": "val",
        **val_metrics,
    })
    all_summary_rows.append({
        "dataset": dataset_name,
        "split": "test",
        **test_metrics,
    })

    val_conf = confusion_matrix(val_labels, np.argmax(val_logits, axis=-1)).tolist()
    test_conf = confusion_matrix(test_labels, np.argmax(test_logits, axis=-1)).tolist()

    all_confusions[f"{dataset_name}_val"] = val_conf
    all_confusions[f"{dataset_name}_test"] = test_conf

    val_report = classification_report(
        val_labels,
        np.argmax(val_logits, axis=-1),
        output_dict=True,
        zero_division=0,
    )
    test_report = classification_report(
        test_labels,
        np.argmax(test_logits, axis=-1),
        output_dict=True,
        zero_division=0,
    )

    def flatten_report(report_dict, dataset_name, split_name):
        rows = []
        for label_or_avg, vals in report_dict.items():
            if isinstance(vals, dict):
                row = {
                    "dataset": dataset_name,
                    "split": split_name,
                    "label_or_avg": str(label_or_avg),
                }
                row.update(vals)
                rows.append(row)
        return rows

    all_report_rows.extend(flatten_report(val_report, dataset_name, "val"))
    all_report_rows.extend(flatten_report(test_report, dataset_name, "test"))

    all_run_logs.append({
        "dataset": dataset_name,
        "model_name": MODEL_NAME,
        "epsilon": EPSILON,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "seed": SEED,
        "use_normalization": USE_NORMALIZATION,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "train_runtime_sec": train_runtime_sec,
        "checkpoint_dir": str(checkpoint_dir),
        "history_path": str(history_path),
        "run_finished_at": datetime.now().isoformat(timespec="seconds"),
    })

    print(f"Completed {dataset_name}: val_macro_f1={val_metrics['macro_f1']:.4f} | test_macro_f1={test_metrics['macro_f1']:.4f}")



RUNNING FGM DATASET: ben_sarc_binary
Loaded shapes -> train: (20508, 4) | val: (2564, 4) | test: (2564, 4)
Detected columns -> train: text='text', label='label_binary' | val: text='text', label='label_binary' | test: text='text', label='label_binary'


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 51610.11it/s]
ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

Starting FGM training...


Epoch 1/3 | train_loss=0.5295 | val_acc=0.7676 | val_macro_f1=0.7647 | val_f1_binary=0.7391


Epoch 2/3 | train_loss=0.3528 | val_acc=0.8023 | val_macro_f1=0.8022 | val_f1_binary=0.8048


...

Epoch 1/3 | train_loss=0.5885 | val_acc=0.7643 | val_macro_f1=0.7639 | val_f1_binary=0.7742


Epoch 2/3 | train_loss=0.4210 | val_acc=0.7843 | val_macro_f1=0.7842 | val_f1_binary=0.7813


Epoch 3/3 | train_loss=0.2415 | val_acc=0.7880 | val_macro_f1=0.7880 | val_f1_binary=0.7901


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.17it/s]


Completed banglasarc3_binary: val_macro_f1=0.7880 | test_macro_f1=0.7693


In [11]:

# =========================
# SAVE GLOBAL OUTPUTS
# =========================

summary_df = pd.DataFrame(all_summary_rows).sort_values(["dataset", "split"]).reset_index(drop=True)
reports_df = pd.DataFrame(all_report_rows).sort_values(["dataset", "split", "label_or_avg"]).reset_index(drop=True)
label_stats_df = pd.DataFrame(all_label_stats).sort_values(["dataset", "split"]).reset_index(drop=True)
run_logs_df = pd.DataFrame(all_run_logs).sort_values(["dataset"]).reset_index(drop=True)

summary_path = TABLES / "06_fgm_banglabert_binary_summary_all_datasets.csv"
confusion_path = TABLES / "06_fgm_banglabert_binary_confusion_matrices_all_datasets.json"
reports_path = REPORTS / "06_fgm_banglabert_binary_classification_reports_all_datasets.csv"
label_stats_path = TABLES / "06_fgm_binary_dataset_label_stats.csv"
run_logs_path = RUN_LOGS / "06_fgm_banglabert_run_metadata.csv"

summary_df.to_csv(summary_path, index=False)
with open(confusion_path, "w", encoding="utf-8") as f:
    json.dump(all_confusions, f, ensure_ascii=False, indent=2)

reports_df.to_csv(reports_path, index=False)
label_stats_df.to_csv(label_stats_path, index=False)
run_logs_df.to_csv(run_logs_path, index=False)

print("Saved global files:")
print("-", summary_path)
print("-", confusion_path)
print("-", reports_path)
print("-", label_stats_path)
print("-", run_logs_path)


Saved global files:
- ../04_outputs/tables/06_fgm_banglabert_binary_summary_all_datasets.csv
- ../04_outputs/tables/06_fgm_banglabert_binary_confusion_matrices_all_datasets.json
- ../04_outputs/reports/06_fgm_banglabert_binary_classification_reports_all_datasets.csv
- ../04_outputs/tables/06_fgm_binary_dataset_label_stats.csv
- ../04_outputs/run_logs/06_fgm_banglabert_run_metadata.csv


In [12]:

# =========================
# DISPLAY RESULTS
# =========================

print("Summary metrics:")
display(summary_df)

print("\nDataset label stats:")
display(label_stats_df)

print("\nRun metadata:")
display(run_logs_df)


Summary metrics:


,dataset,split,accuracy,precision_binary,recall_binary,f1_binary,macro_f1
0,banglasarc3_binary,test,0.769327,0.764706,0.778055,0.771323,0.769309
1,banglasarc3_binary,val,0.788030,0.782396,0.798005,0.790123,0.788009
2,banglasarc_binary,test,0.986328,0.994764,0.969388,0.981912,0.985462
3,banglasarc_binary,val,0.990215,0.994792,0.979487,0.987080,0.989603
4,ben_sarc_binary,test,0.809672,0.810642,0.808112,0.809375,0.809672
5,ben_sarc_binary,val,0.802262,0.794677,0.815133,0.804775,0.802229



Dataset label stats:


,dataset,split,text_column,label_column,n_rows,label_0_count,label_0_pct,label_1_count,label_1_pct
0,banglasarc3_binary,test,text,label_binary,802,401,50.00,401,50.00
1,banglasarc3_binary,train,text,label_binary,6413,3207,50.01,3206,49.99
2,banglasarc3_binary,val,text,label_binary,802,401,50.00,401,50.00
3,banglasarc_binary,test,text,label_binary,512,316,61.72,196,38.28
4,banglasarc_binary,train,text,label_binary,4089,2527,61.80,1562,38.20
5,banglasarc_binary,val,text,label_binary,511,316,61.84,195,38.16
6,ben_sarc_binary,test,text,label_binary,2564,1282,50.00,1282,50.00
7,ben_sarc_binary,train,text,label_binary,20508,10254,50.00,10254,50.00
8,ben_sarc_binary,val,text,label_binary,2564,1282,50.00,1282,50.00



Run metadata:


,dataset,model_name,epsilon,epochs,batch_size,max_length,learning_rate,weight_decay,warmup_ratio,seed,use_normalization,best_epoch,best_val_macro_f1,train_runtime_sec,checkpoint_dir,history_path,run_finished_at
0,banglasarc3_binary,csebuetnlp/banglabert,0.5,3,8,128,0.00002,0.01,0.1,42,False,3,0.788009,2860.614211,../03_models/checkpoints/06_fgm_banglasarc3_bi...,../04_outputs/run_logs/06_fgm_banglasarc3_bina...,2026-03-27T06:42:36
1,banglasarc_binary,csebuetnlp/banglabert,0.5,3,8,128,0.00002,0.01,0.1,42,False,3,0.989603,1877.768989,../03_models/checkpoints/06_fgm_banglasarc_binary,../04_outputs/run_logs/06_fgm_banglasarc_binar...,2026-03-27T05:54:09
2,ben_sarc_binary,csebuetnlp/banglabert,0.5,3,8,128,0.00002,0.01,0.1,42,False,2,0.802229,9461.328741,../03_models/checkpoints/06_fgm_ben_sarc_binary,../04_outputs/run_logs/06_fgm_ben_sarc_binary_...,2026-03-27T05:22:12


In [13]:

# =========================
# QUICK ERROR SNAPSHOT
# =========================

example_error_tables = {}

for dataset_name in DATASETS:
    pred_path = PREDICTIONS / f"06_fgm_banglabert_{dataset_name}_test_predictions.csv"
    if pred_path.exists():
        dfp = pd.read_csv(pred_path)
        err = dfp[dfp["correct"] == 0].copy().sort_values("confidence", ascending=False).head(10)
        example_error_tables[dataset_name] = err

for dataset_name, err_df in example_error_tables.items():
    print("\n" + "=" * 80)
    print(f"Top confident test errors: {dataset_name}")
    print("=" * 80)
    display(
        err_df[
            [
                "text",
                "gold_label",
                "pred_label",
                "prob_label_0",
                "prob_label_1",
                "confidence",
            ]
        ]
    )



Top confident test errors: ben_sarc_binary


,text,gold_label,pred_label,prob_label_0,prob_label_1,confidence
2210,ইয়ার্কির একটা সীমা থাকা উচিৎ,1,0,0.987847,0.012153,0.987847
561,ভালা লাগছে তুমি তাদের মনে রেখছো বলে,1,0,0.986883,0.013117,0.986883
1202,ঈদ শুধু আপনাদের জন্যই !,1,0,0.984649,0.015351,0.984649
991,অসম্ভব ভাল ছবি । এক কথায় অনবদ্য । বহুদিন পরে ...,1,0,0.982532,0.017468,0.982532
2004,যেহেতু ভাইরাস বাতাসে ভেসে মানুষকে আক্রান্ত করছ...,1,0,0.981684,0.018316,0.981684
1198,একটু দেরি হওয়াতে মৃত্যুর মুখে পতিত হচ্ছে বা হব...,1,0,0.979946,0.020054,0.979946
621,পোশাক মানুষের আচার আচরণ এবং পরিবারিক চাল চলন ব...,1,0,0.979789,0.020211,0.979789
817,চোটের কারণে দলের বাইরে চলে গেছেন এবি ডি ভিলিয়...,1,0,0.979232,0.020768,0.979232
2454,ঊনিশশো বাইশ এর পর ঊনিশশো তেইশ থেকে শুরু করে ঊন...,1,0,0.975792,0.024208,0.975792
868,আপনার দৌড় কতটুকু জানা আছে । এই লাইন গুলোর কোন...,1,0,0.971312,0.028688,0.971312



Top confident test errors: banglasarc_binary


,text,gold_label,pred_label,prob_label_0,prob_label_1,confidence
286,একটি বিপন্ন প্রজাতিতে শুভেচ্ছা,1,0,0.996864,0.003136,0.996864
336,তুমি না করলে কোনো কিছুই কাজে আসবে না.,1,0,0.987890,0.012110,0.987890
298,"আপনারা সবাই একে অপরকেও জানেন, তাই না?",1,0,0.935852,0.064148,0.935852
302,অটোকারেক্ট এখনও মনে করে আমি দিনে 12 বার ‘হাঁস’...,1,0,0.904795,0.095205,0.904795
107,"জাই হক , আমি কিছু জ্ঞান চিখলাম ধন্যবাদ",0,1,0.312137,0.687863,0.687863
67,প্রতিটি ক্যালেন্ডারের দিনগুলি সংখ্যাযুক্ত,1,0,0.642660,0.357340,0.642660
418,অ্যালকোহল ‘প্রেরণ’ বোতামের আকার 89%বৃদ্ধি করে।,1,0,0.545341,0.454659,0.545341



Top confident test errors: banglasarc3_binary


,text,gold_label,pred_label,prob_label_0,prob_label_1,confidence
247,মনোযোগ দিয়ে প্রতিটা কথা শুনুন এবং সে অনুযায়ী...,1,0,0.988345,0.011655,0.988345
553,আপনার পোষ্ট গুলো অসাধারণ সবসময় সত্য কথা তুলে ধ...,1,0,0.986160,0.013840,0.986160
450,আমাদের সন্তানেরা একদিন বিশ্ব জয় করবে স্যার,1,0,0.985149,0.014851,0.985149
159,ধর্ম গ্রন্থের রেফারেন্স টানার আগে ঘটনার প্রেক্...,1,0,0.985136,0.014864,0.985136
158,ধন্যবাদ নিজের পরিচয় দেওয়ার জন্য,1,0,0.982733,0.017267,0.982733
461,হ্যাতের ভাষণ দুনিয়ার সবচেয়ে বিরক্তিকর জিনিস রি...,1,0,0.981506,0.018494,0.981506
570,যে যেই যায়গার লোক তার টার্গেট তো সেখানেই থাকবে,1,0,0.980150,0.019850,0.980150
695,ইতিহাসের পাতায় লেখা থাকবে ৭১এর পরাজিত শক্তি মা...,1,0,0.979620,0.020380,0.979620
325,১৬ বছর পর প্রভু চেঞ্জ হইছে আপনার ১৬ বছরের অভ্য...,1,0,0.976235,0.023765,0.976235
228,এখন তো ফেরেস্তারা ক্ষমতায় তাহলে আলু ৮০ টাহা কে...,1,0,0.972988,0.027012,0.972988
